# Fine-Tuning SraVaani for Low-Resource Indian Languages: A Wancho Case Study

*Adapting a 65-language FastConformer speech recognizer to a language it barely knows — either by training only the decoder heads, or by fine-tuning the whole model.*

This notebook walks through fine-tuning [SraVaani](https://huggingface.co/ARTPARK-IISc/SraVaani-1.0) — a Hybrid RNN-T + CTC FastConformer ASR model pretrained on 65 Indian languages — on **Wancho**, a low-resource Tibeto-Burman language spoken in Arunachal Pradesh, India.

**TL;DR**
- Base model: [SraVaani](https://huggingface.co/ARTPARK-IISc/SraVaani-1.0) Hybrid RNN-T + CTC BPE FastConformer, 65 Indian languages, from ARTPARK-IISc
- Target language: Wancho (not well represented in the pretraining mix)
- Data: [`ARTPARK-IISc/Vaani-transcription-part`](https://huggingface.co/datasets/ARTPARK-IISc/Vaani-transcription-part) on the Hugging Face Hub, Wancho config
- Two fine-tuning modes, chosen with one switch, `FULL_FINETUNE`, in §5:
  - `False` — **encoder frozen**: only the decoder, joint network and CTC head are trained
  - `True` — **full fine-tuning**: every parameter, encoder included, is trained
- Hardware: 1x NVIDIA T4 (16GB VRAM) is enough for the frozen-encoder mode; full fine-tuning is tight on 16GB — see §1
- Result on the Wancho test set after 2 epochs (§10): base 63.89% WER → encoder frozen 63.52%, full fine-tuning **63.44%** (with the encoder's BatchNorm statistics frozen)

Everything that `FULL_FINETUNE` changes is tagged in the code with a `# [FULL-FT]` comment that explains why, so a search for `[FULL-FT]` shows every place the two modes differ.

## 1. Before You Begin

You'll need:

- A CUDA-capable GPU. We used a **Tesla T4 (16GB VRAM)** — the smallest end of what's workable for RNN-T fine-tuning.
  - **Encoder frozen:** fits comfortably on 16GB.
  - **Full fine-tuning:** considerably heavier — on top of the weights, every encoder parameter now also carries a gradient and two AdamW moment buffers, and encoder activations must be kept for the backward pass. In our Wancho runs peak memory was **8.5GB** with the encoder frozen (fp32, batch 4) and **12.0GB** for full fine-tuning (bf16, batch 2), so 16GB is the floor and a 24GB+ GPU gives you headroom. The training cell prints peak GPU memory at every validation pass, so check it after the first epoch and lower `BATCH_SIZE` (raising `ACCUMULATE` to keep the effective batch) if you are near the limit.
- At least a few GB of free disk (checkpoint is ~1.8GB, the Wancho dataset is ~1.4GB; with full fine-tuning each `.ckpt` saved by `exp_manager` is several GB because it includes optimizer state for all parameters).
- A Hugging Face account + token with access to `ARTPARK-IISc/Vaani-transcription-part`.

## 2. Environment Setup

Create a virtual environment and install NeMo with ASR support:

```bash
python3 -m venv venv
source venv/bin/activate
pip install "nemo_toolkit[asr,cu13]" pytorch-lightning omegaconf
```

> **Gotcha:** `pip install nemo_toolkit[asr]` alone is not enough. NeMo's RNN-T/TDT loss needs the `numba-cuda` GPU kernel backend, which only comes via the `cu12`/`cu13` extra (match your CUDA driver version from `nvidia-smi`). Without it, everything works until the first training step, which fails deep in a numba kernel compile with an unhelpful `TypeError: Signature mismatch` error. Fix: `pip install "nemo_toolkit[asr,cu13]"`.

In [ ]:
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

import nemo
print("nemo_toolkit:", nemo.__version__)

# Confirms the RNN-T CUDA kernel backend is actually present —
# if this import fails, you hit the cu12/cu13 gotcha above.
import numba_cuda
print("numba-cuda: OK")

## 3. Download the Pretrained Checkpoint

[Download `SraVaani-nemo-checkpoint.nemo` from Google Drive](https://drive.google.com/drive/folders/1QdMG0R3oGgOThm7uqYeMnt8zGOSx467x) (~1.8GB) and place it in the project root. Don't have access? [Request access here](https://script.google.com/macros/s/AKfycbyAC0vGcGSbu55LNKOn_hY3TsPy8CyF1nasLJULWMrzCdksBJG33bUfLbiKee_m_aKH-w/exec) — access will be shared ASAP.

**Always verify the download.** A `.nemo` file is just a tar archive; a truncated or interrupted download still produces a file that *looks* plausible by size but fails the moment NeMo tries to read past the point where the download died:

```python
import tarfile
with tarfile.open("SraVaani-nemo-checkpoint.nemo") as tf:
    print(tf.getnames())  # raises tarfile.ReadError on a truncated file
```

If this raises `ReadError: unexpected end of data`, re-download — don't debug the training script first. We lost time doing exactly that before checking the checkpoint itself.

## 4. Prepare the Dataset

This notebook expects data in NeMo's tarred format: a **tarred audio archive** (`shard_NNNN.tar`) paired with a **JSONL manifest** (`shard_NNNN.json`), one line per clip, audio 16kHz mono 16-bit PCM:

```json
{"audio_filepath": "train_00000.wav", "text": "pha ama gaga aga gih pan tega mai tai", "duration": 5.547}
```

`audio_filepath` is the filename **inside** the tar, not a filesystem path.

Prepare `train`/`validation`/`test` splits in this format for your own use case before continuing — how you get there depends on where your data lives (a Hugging Face dataset, local files, etc.).

### Cleaning the transcripts

Run every transcript through `clean()` **before** writing the manifests, so the model is trained on speech-only text, and use it again in §9 on both references and hypotheses, so scoring measures transcription rather than formatting.

The `clean()` below is written for the **Vaani** transcripts used here, which carry annotation markup that is not speech: `{...}` glosses after code-switched words (`light {light} bulb {bulb}`), `<pause>`-style tags, `[unintelligible]`/`(...)` notes, and punctuation. **Modify `clean()` according to your dataset** — keep the steps that match your transcription conventions, drop the ones that don't apply, and add any normalization your transcripts need (for example, numerals or script-specific punctuation). Whatever you change, use the same `clean()` for training text and for scoring.

`BRACES` picks what happens to a `word {gloss}` pair:
- `"native"` keeps the transcribed word and drops the gloss — the **native-script** form;
- `"replace"` keeps the Latin gloss in place of the word — the **code-switched** form.

In [ ]:
import re
import string

BRACES = "native"  # "native": keep the word, drop its {gloss}; "replace": swap the word for its {gloss}


def clean(text):
    # Written for Vaani transcripts — modify these steps according to your dataset.
    if not isinstance(text, str):
        text = ""
    if BRACES == "replace":
        # keep the Latin gloss, drop the transliterated token: the CODE-SWITCHED form
        text = re.sub(r"(?:\S+\s*)?\{([^}]*)\}", lambda m: " " + m.group(1) + " ", text)
    else:
        # drop the gloss, keep the transliteration: the NATIVE-SCRIPT form
        text = re.sub(r"\{[^}]*\}", "", text)
    text = re.sub(r"[\(\[].*?[\)\]]", "", text)  # (...) and [...] annotations
    text = re.sub(r"<[^>]+>", "", text)          # <pause>-style tags
    # separators and sentence punctuation become spaces, remaining punctuation is dropped
    text = text.replace("·", " ")
    text = (text.replace("।", " ").replace("|", " ").replace("-", " ")
                .replace(".", " ").replace(",", " ").replace("\n", " "))
    text = text.translate(str.maketrans("", "", string.punctuation + "।॥“”‘’–—"))
    return re.sub(" +", " ", text).strip()


print(clean("light {light} bulb {bulb} ima <pause> yellow {yellow} colour {colour} ma [inaudible]."))

## 5. Load the Pretrained Model and Choose the Fine-Tuning Mode

Set `FULL_FINETUNE` in the cell below. It is the only line you need to change; every later cell reads it.

### `FULL_FINETUNE = False` — encoder frozen

With ~5-9 hours of Wancho audio against a model pretrained on 65 languages, updating the encoder risks catastrophic forgetting of everything it already knows about general Indian-language phonetics. **Freezing the encoder and only training the decoder + joint network** is the safer starting point — it adapts the parts of the model that are language-specific (the prediction network and output vocabulary mapping) while leaving the acoustic feature extraction untouched. It is also cheap: AdamW only holds optimizer state for the small heads, so it runs comfortably on a 16GB GPU.

Choose this mode when your data is small, your domain shift is mostly about vocabulary rather than acoustics, or you are first confirming that the pipeline works end to end.

### `FULL_FINETUNE = True` — full fine-tuning

**Every parameter is trained, encoder included.** This lets the model adapt its acoustic representations, which matters when the target language's phonetics differ enough from the pretraining mix that a frozen encoder becomes the bottleneck. The risk is still catastrophic forgetting, so instead of freezing, the notebook keeps encoder updates gentle:

- the encoder gets a **10× lower learning rate** than the heads (§6), so it moves slowly while the decoder/joint adapt at the usual fine-tuning rate;
- a **longer warmup** (§6) keeps the first updates small while AdamW's moment estimates are still noisy;
- the encoder's **BatchNorm statistics are frozen** (§7). We do this because this notebook demonstrates fine-tuning with a small per-step batch (2 clips, so it fits on a 16GB GPU), and BatchNorm statistics estimated from 2 clips are too noisy to replace the ones learned in pretraining. The BatchNorm layers stay in eval mode and keep normalizing with the pretrained statistics, while their weights still train. With a much larger per-step batch you may not need this;
- **mixed precision** and a **smaller per-step batch** (§6, §7) make room for the encoder's gradients, optimizer state and activations in GPU memory.

Choose this mode when you have more data, when the language sounds unlike anything in the pretraining mix, or when the frozen-encoder mode has stopped improving. Use the learning rates recommended in §6.

In full fine-tuning mode the cell calls `model.encoder.unfreeze()` defensively, in case you restore a checkpoint that was saved with the encoder frozen.

In [ ]:
from nemo.collections.asr.models import EncDecHybridRNNTCTCBPEModel

NEMO_CHECKPOINT = "SraVaani-nemo-checkpoint.nemo"

# [FULL-FT] The one switch: False = encoder frozen, True = fine-tune every parameter.
FULL_FINETUNE = False

model = EncDecHybridRNNTCTCBPEModel.restore_from(NEMO_CHECKPOINT)

if FULL_FINETUNE:
    # [FULL-FT] Leave the encoder trainable. unfreeze() is a no-op on the released
    # checkpoint, but guards against restoring one saved with a frozen encoder.
    model.encoder.unfreeze()
else:
    model.encoder.freeze()

n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"FULL_FINETUNE={FULL_FINETUNE}: training {n_trainable / 1e6:.1f}M of {n_total / 1e6:.1f}M parameters")

## 6. Configure Data Loaders and Optimizer

A few choices here are tuned specifically for a small, cheap GPU rather than "what NeMo defaults to". Data loading is the same in both modes; with `FULL_FINETUNE = True` three things change:

- **Batch size 2 × accumulation 16** instead of 4 × 8. The effective batch is still 32, so the optimization behaves the same, but each forward/backward pass holds half as many clips' worth of encoder activations.
- **Discriminative learning rates.** NeMo supports per-module learning rates natively through `model.cfg.optim_param_groups`: any top-level module named there (here `encoder`) gets its own optimizer param group with the settings given, and every other parameter (decoder, joint, CTC decoder) falls into a default group that uses `model.cfg.optim.lr`. The cosine schedule scales each group from its own peak LR, so the encoder stays at 1/10 of the heads' LR throughout.
- **Warmup = 10% of total steps** instead of a fixed 10 steps.

The cell prints the optimizer's param groups at the end, so you can confirm which parameters are being trained and at what learning rate.

### Recommended learning rates

| Mode | Decoder + joint + CTC head (`LR`) | Encoder (`ENCODER_LR`) |
|---|---|---|
| `FULL_FINETUNE = False` | `1e-4` | frozen |
| `FULL_FINETUNE = True` | `1e-4` | `1e-5` |

These are the defaults in the cell below. Keep the heads at `1e-4` even in full fine-tuning — they are the part of the model that has to learn the new language, and in our Wancho runs, lowering their LR together with the encoder's made full fine-tuning clearly worse. `ENCODER_LR = 1e-5` keeps the encoder 10× slower than the heads; `2e-5` performed about the same.

With full fine-tuning `min_lr` drops from `1e-6` to `1e-7`, so that it stays below `ENCODER_LR` and the encoder's learning rate still decays over the run.

In [ ]:
from omegaconf import OmegaConf, open_dict

TRAIN_SHARDS, TRAIN_MANIFESTS = "data/train/shard_0000.tar", "data/train/shard_0000.json"
VAL_SHARDS, VAL_MANIFESTS = "data/validation/shard_0000.tar", "data/validation/shard_0000.json"

MAX_EPOCHS = 2
NUM_WORKERS = 1    # kept small — see the memory section below
if FULL_FINETUNE:
    # [FULL-FT] Encoder activations are now kept for the backward pass, so halve the
    # per-step batch and double accumulation: effective batch stays 2 * 16 = 32.
    BATCH_SIZE, ACCUMULATE = 2, 16
else:
    BATCH_SIZE, ACCUMULATE = 4, 8  # kept small — effective batch size = 4 * 8 = 32

LR = 1e-4          # decoder + joint + CTC head (and the only LR when the encoder is frozen) — recommended
ENCODER_LR = 1e-5  # [FULL-FT] full fine-tuning only — recommended; lower values did worse (see above)

with open(TRAIN_MANIFESTS, encoding="utf-8") as f:
    n_train = sum(1 for line in f if line.strip())

# batches/epoch (ceil, dataloader granularity) -> optimizer steps/epoch (// ACCUMULATE) -> total
MAX_STEPS = (-(-n_train // BATCH_SIZE) // ACCUMULATE) * MAX_EPOCHS
assert MAX_STEPS > 0, (f"{n_train} clips is less than one optimizer step "
                       f"({BATCH_SIZE * ACCUMULATE} clips) — lower ACCUMULATE")
# [FULL-FT] 10% of the run when the encoder trains; a fixed 10 steps is enough for the heads alone.
WARMUP_STEPS = max(1, MAX_STEPS // 10) if FULL_FINETUNE else 10
print(f"{n_train} train clips -> {MAX_STEPS} total optimizer steps over {MAX_EPOCHS} epochs "
      f"({WARMUP_STEPS} warmup)")

with open_dict(model.cfg):
    model.cfg.train_ds.manifest_filepath = TRAIN_MANIFESTS
    model.cfg.train_ds.tarred_audio_filepaths = TRAIN_SHARDS
    model.cfg.train_ds.is_tarred = True
    model.cfg.train_ds.shuffle_n = 2048
    model.cfg.train_ds.batch_size = BATCH_SIZE
    model.cfg.train_ds.num_workers = NUM_WORKERS
    model.cfg.train_ds.pin_memory = True

    model.cfg.validation_ds.manifest_filepath = VAL_MANIFESTS
    model.cfg.validation_ds.tarred_audio_filepaths = VAL_SHARDS
    model.cfg.validation_ds.is_tarred = True
    model.cfg.validation_ds.batch_size = BATCH_SIZE
    model.cfg.validation_ds.num_workers = NUM_WORKERS

    # Keep LR small — we're nudging a pretrained model, not training from scratch
    model.cfg.optim.lr = LR
    model.cfg.optim.name = "adamw"
    model.cfg.optim.weight_decay = 1e-3
    if FULL_FINETUNE:
        # [FULL-FT] Discriminative LR: the encoder gets its own param group at ENCODER_LR;
        # everything not listed here (decoder, joint, ctc_decoder) uses optim.lr.
        model.cfg.optim_param_groups = {"encoder": {"lr": ENCODER_LR}}
    model.cfg.optim.sched = OmegaConf.create({
        "name": "CosineAnnealing",
        "warmup_steps": WARMUP_STEPS,
        "max_steps": MAX_STEPS,
        # [FULL-FT] min_lr must stay below ENCODER_LR, or the encoder's LR never decays
        "min_lr": 1e-7 if FULL_FINETUNE else 1e-6,
    })

model.setup_training_data(model.cfg.train_ds)
model.setup_validation_data(model.cfg.validation_ds)
model.setup_optimization(model.cfg.optim)

for g in model._optimizer.param_groups:
    n_params = sum(p.numel() for p in g["params"] if p.requires_grad)
    print(f"param group: {n_params / 1e6:.1f}M trainable params, peak lr {g['initial_lr']:.0e}")

## 7. Train

A couple of additions beyond a bare `trainer.fit(model)`:

- A small `MetricsLogger` callback prints `train_loss` every step, and `val_wer` plus **peak GPU memory** at every validation pass — useful when you're watching a run live rather than only reviewing TensorBoard after the fact, and the number to check before raising or lowering `BATCH_SIZE`.
- `exp_manager` handles checkpointing (top-3 by `val_wer`) and TensorBoard logging. Each mode gets its own experiment directory and output `.nemo`, so running one never overwrites the other.
- **Frozen BatchNorm statistics (full fine-tuning only).** The FastConformer encoder normalizes inside every conformer block with BatchNorm. If the encoder trains with those layers in training mode, they normalize each 2-clip batch with that batch's own statistics and overwrite their pretrained running averages — the decoder then learns on features that differ from what it sees at test time, and the fine-tuned model starts dropping words. The `FreezeEncoderBatchNorm` callback keeps the encoder's BatchNorm layers in eval mode on every training batch: their weights and biases still train, but they normalize with the pretrained statistics, which stay exactly as they were.
- **Precision.** The frozen-encoder mode trains in full fp32. With `FULL_FINETUNE = True` the trainer switches to mixed precision: `bf16-mixed` on GPUs that support bfloat16 (compute capability 8.0+: A100, L4, RTX 30xx/40xx, H100) and `16-mixed` (fp16 with loss scaling) on older ones such as the T4. NeMo's RNN-T loss casts its inputs back to fp32 internally, so the loss itself stays numerically safe either way. With `16-mixed`, expect the first few optimizer steps to be skipped while the loss scaler calibrates — that is normal.

In [ ]:
import lightning.pytorch as pl
from nemo.utils.exp_manager import exp_manager

if FULL_FINETUNE:
    # [FULL-FT] Mixed precision makes room for encoder gradients and optimizer state —
    # bf16 where the hardware supports it, fp16 otherwise.
    PRECISION = "bf16-mixed" if torch.cuda.get_device_capability()[0] >= 8 else "16-mixed"
    RUN_NAME = "sra-vaani-full-finetune"  # [FULL-FT] own experiment dir and output file
else:
    PRECISION = "32-true"
    RUN_NAME = "sra-vaani-finetune"
FINETUNED_CHECKPOINT = RUN_NAME.replace("finetune", "finetuned") + ".nemo"
print(f"Training precision: {PRECISION} | experiment: experiments/{RUN_NAME} | output: {FINETUNED_CHECKPOINT}")


class MetricsLogger(pl.Callback):
    """Prints train loss every step, and val WER + peak GPU memory at every eval step."""

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        loss = trainer.callback_metrics.get("train_loss")
        if loss is not None:
            print(f"[step {trainer.global_step}] train_loss={loss.item():.4f}")

    def on_validation_epoch_end(self, trainer, pl_module):
        wer = trainer.callback_metrics.get("val_wer")
        if wer is not None:
            print(f"[step {trainer.global_step}] val_wer={wer.item():.4f}")
        peak_gb = torch.cuda.max_memory_allocated() / 2**30
        total_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
        print(f"[step {trainer.global_step}] peak GPU memory {peak_gb:.1f} / {total_gb:.1f} GB")


class FreezeEncoderBatchNorm(pl.Callback):
    """[FULL-FT] Keep the encoder's BatchNorm layers in eval mode while the encoder trains.

    Their weights and biases still update, but they normalize with the pretrained
    running statistics instead of the statistics of each small training batch, and
    those running statistics are never overwritten.
    """

    def on_train_batch_start(self, trainer, pl_module, batch, batch_idx):
        # Lightning puts the whole model back in train mode after every validation
        # pass, so re-apply eval mode on every batch (cheap: one loop over modules).
        for m in pl_module.encoder.modules():
            if isinstance(m, torch.nn.modules.batchnorm._BatchNorm):
                m.eval()


callbacks = [MetricsLogger()]
if FULL_FINETUNE:
    callbacks.append(FreezeEncoderBatchNorm())  # [FULL-FT]

trainer = pl.Trainer(
    devices=1,
    accelerator="gpu",
    precision=PRECISION,
    max_epochs=MAX_EPOCHS,
    accumulate_grad_batches=ACCUMULATE,
    gradient_clip_val=1.0,
    val_check_interval=1.0,
    log_every_n_steps=10,
    logger=False,
    enable_checkpointing=False,
    callbacks=callbacks,
)

exp_manager(trainer, {
    "exp_dir": "experiments/",
    "name": RUN_NAME,
    "create_tensorboard_logger": True,
    "create_checkpoint_callback": True,
    "checkpoint_callback_params": {"monitor": "val_wer", "mode": "min", "save_top_k": 3},
})

trainer.fit(model)
model.save_to(FINETUNED_CHECKPOINT)

## 8. Monitor Training

Everything logged during `trainer.fit()` also lands in the TensorBoard event file under `experiments/`, which we can pull directly with `EventAccumulator` for a quick static plot — handy in a notebook where you don't want to launch a separate TensorBoard server.

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import glob
import matplotlib.pyplot as plt

event_file = sorted(glob.glob(f"experiments/{RUN_NAME}/*/events.out.tfevents.*"))[-1]
ea = EventAccumulator(event_file, size_guidance={"scalars": 0})
ea.Reload()

train_loss = ea.Scalars("train_loss")
steps = [e.step for e in train_loss]
values = [e.value for e in train_loss]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(steps, values, linewidth=2)
ax.set_xlabel("Optimizer step")
ax.set_ylabel("train_loss")
ax.set_title(f"SraVaani fine-tune ({'full' if FULL_FINETUNE else 'encoder frozen'}) — training loss (Wancho)")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Evaluate: Original vs. Fine-Tuned

The real test is comparing the **original** and **fine-tuned** checkpoints on the same held-out test set, with the same text cleaning applied to both references and hypotheses before scoring — otherwise you're partly measuring formatting differences, not transcription quality.

The fine-tuned checkpoint is whichever one §7 just wrote, for the mode you chose. References and hypotheses both go through the same `clean()` defined in §4.

In [ ]:
import csv
import json
import shutil
import tarfile
import tempfile

import jiwer

TEST_SHARDS, TEST_MANIFEST = "data/test/shard_0000.tar", "data/test/shard_0000.json"
ORIGINAL_CHECKPOINT = "SraVaani-nemo-checkpoint.nemo"  # FINETUNED_CHECKPOINT comes from §7
INFER_BATCH_SIZE = 32


def load_test_set(shards_path, manifest_path):
    tmp_dir = tempfile.mkdtemp(prefix="wancho_test_")
    with tarfile.open(shards_path) as tar:
        tar.extractall(tmp_dir)
    audio_filepaths, references = [], []
    with open(manifest_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                entry = json.loads(line)
                audio_filepaths.append(f"{tmp_dir}/{entry['audio_filepath']}")
                references.append(entry["text"])
    return audio_filepaths, references, tmp_dir


def run_inference(checkpoint_path, audio_filepaths):
    m = EncDecHybridRNNTCTCBPEModel.restore_from(checkpoint_path)
    m.eval()
    if torch.cuda.is_available():
        m = m.cuda()
    hyps = m.transcribe(audio=audio_filepaths, batch_size=INFER_BATCH_SIZE)
    hyps = [h.text if hasattr(h, "text") else h for h in hyps]
    del m
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return hyps


# Free the training model before loading checkpoints for inference.
del model, trainer
torch.cuda.empty_cache()

audio_filepaths, references, tmp_dir = load_test_set(TEST_SHARDS, TEST_MANIFEST)
print(f"Loaded {len(audio_filepaths)} test clips")

hyp_original = run_inference(ORIGINAL_CHECKPOINT, audio_filepaths)
hyp_finetuned = run_inference(FINETUNED_CHECKPOINT, audio_filepaths)
shutil.rmtree(tmp_dir, ignore_errors=True)

# References and both sets of hypotheses go through the same clean() from §4
refs_clean = [clean(r) for r in references]
hyp_original_clean = [clean(h) for h in hyp_original]
hyp_finetuned_clean = [clean(h) for h in hyp_finetuned]

wer_original = jiwer.wer(refs_clean, hyp_original_clean)
wer_finetuned = jiwer.wer(refs_clean, hyp_finetuned_clean)

print(f"Original checkpoint  WER: {wer_original:.4f} ({wer_original * 100:.2f}%)")
print(f"Fine-tuned checkpoint WER: {wer_finetuned:.4f} ({wer_finetuned * 100:.2f}%)")

with open(f"inference_results_{RUN_NAME}.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["audio_filepath", "reference", "hyp_original", "hyp_finetuned"])
    for path, ref, ho, hf in zip(audio_filepaths, refs_clean, hyp_original_clean, hyp_finetuned_clean):
        writer.writerow([path.rsplit("/", 1)[-1], ref, ho, hf])

## 10. Results

All numbers are on the Wancho **test** split of `ARTPARK-IISc/Vaani-transcription-part` (1,952 clips, 2.1 hours), scored with the `clean()` from §4 applied to both references and hypotheses. Both models were trained on the Wancho train split (7,144 clips, 7.6 hours) for 2 epochs with the recommended learning rates from §6 and an effective batch of 32, on a single GPU.

| Model | WER | CER |
|---|---|---|
| Original SraVaani (zero-shot on Wancho) | 63.89% | 21.29% |
| `FULL_FINETUNE = False` — encoder frozen (`LR = 1e-4`) | 63.52% | 21.55% |
| `FULL_FINETUNE = True` — full fine-tuning (`LR = 1e-4`, `ENCODER_LR = 1e-5`), encoder BatchNorm statistics frozen | **63.44%** | 21.36% |

Both modes improve on the original checkpoint after 2 epochs, and full fine-tuning comes out slightly ahead. The gains are modest, as expected for 2 epochs on a few hours of audio of a language the model saw little of during pretraining.

> **Remark — BatchNorm statistics were frozen for full fine-tuning.** The full fine-tuning result above was trained with the `FreezeEncoderBatchNorm` callback from §7, which keeps the encoder's BatchNorm layers in eval mode: their weights train, but their running statistics stay at the pretrained values. Keep the callback on when you fine-tune the encoder with small per-step batches like the ones in this demonstration.

## 11. Next Steps

- **Train longer.** 2 epochs on a few hours of audio is a short run; validation WER was still falling in both modes, so more epochs should help.
- **Watch for forgetting with full fine-tuning.** Updating the encoder can improve the target language while degrading the other 64. Score the fine-tuned model on a held-out set from one or two pretraining languages and check it against the original checkpoint before you use it as a multilingual model.
- **Repeat for another of the 65 pretraining languages** that's similarly underrepresented — the pipeline here (download → clean → tar/manifest → fine-tune → compare-to-baseline) is language-agnostic; only the data paths and the checkpoint need to change.
- **Try it on your own data.** Everything here is language- and dataset-agnostic once you've prepared your data in the format from §4 — swap in your own audio/transcripts for your own use case (a dialect, a domain, a language SraVaani wasn't pretrained on at all), adapt `clean()` to your transcripts, and the rest of the notebook runs unchanged.

---

*This notebook documents real fine-tuning runs. Feel free to adapt it for your own use case. Questions or clarifications about this notebook? Reach out at agneedh@artpark.in.*

---